# EX — LangChain Real-World Exercises

**Note:** requires `langchain` (`pip install langchain langchain-community --break-system-packages`).
If you don't have API keys configured, focus on reading/understanding the composition
patterns (PromptTemplate, chains, memory) — the concepts transfer directly once you
plug in a real model.


In [ ]:
try:
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    LANGCHAIN_AVAILABLE = True
except ImportError:
    LANGCHAIN_AVAILABLE = False
print("LangChain available:", LANGCHAIN_AVAILABLE)


## 1. PromptTemplate — reusable, parameterized prompts

In [ ]:
if LANGCHAIN_AVAILABLE:
    template = PromptTemplate.from_template(
        "Classify the following support ticket into billing, technical, account, or other.\n\nTicket: {ticket}\nLabel:"
    )
    print(template.format(ticket="I was charged twice for one order"))
else:
    print("Install langchain to run this cell.")


### TODO 1
Create a `PromptTemplate` for summarizing a customer review in one sentence, with a `{review}` variable. Print the formatted prompt for a sample review.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
summary_template = PromptTemplate.from_template(
    "Summarize this customer review in one sentence:\n\n{review}"
)
print(summary_template.format(review="Great product but shipping took forever."))
```
</details>


## 2. LCEL Chains — composing steps with `|`
**Pointer:** think of `|` as Unix-pipe-style composition: prompt -> model -> parser.

In [ ]:
# Pseudocode pattern (works once you plug in a real chat model, e.g. ChatOpenAI or ChatGoogleGenerativeAI):
#
# from langchain_openai import ChatOpenAI
# model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# chain = template | model | StrOutputParser()
# result = chain.invoke({"ticket": "The app won't open"})
#
# The exercise: identify what each of the 3 pipeline stages is responsible for.
print("Stage 1 (PromptTemplate): fills in variables to build the final prompt string")
print("Stage 2 (model): sends the prompt to the LLM and gets a raw message back")
print("Stage 3 (StrOutputParser): extracts plain text from the model's response object")


## 3. Memory — Multi-Turn Conversation State
**Pointer:** unbounded memory grows token cost; summarizing older turns keeps cost flat.

In [ ]:
class SimpleBufferMemory:
    """A minimal stand-in for LangChain's ConversationBufferMemory, for offline practice."""
    def __init__(self):
        self.turns = []
    def add(self, role, content):
        self.turns.append({"role": role, "content": content})
    def render(self):
        return "\n".join(f"{t['role']}: {t['content']}" for t in self.turns)

memory = SimpleBufferMemory()
memory.add("user", "I want to return an order")
memory.add("assistant", "Sure — what's the order number?")
memory.add("user", "98765")
print(memory.render())


### TODO 2
Extend `SimpleBufferMemory` with a `max_turns` limit: once exceeded, collapse everything except the last 2 turns into a single `system` turn reading `'[earlier conversation omitted]'`.

In [ ]:
# TODO


<details><summary>Solution</summary>

```python
class LimitedBufferMemory(SimpleBufferMemory):
    def __init__(self, max_turns=4):
        super().__init__()
        self.max_turns = max_turns
    def add(self, role, content):
        super().add(role, content)
        if len(self.turns) > self.max_turns:
            recent = self.turns[-2:]
            self.turns = [{"role": "system", "content": "[earlier conversation omitted]"}] + recent
```
</details>


## 4. Tools & Agents — Conceptual Exercise
**Pointer:** a tool's docstring/description is what the agent uses to decide when to call it — write it like documentation for a teammate.

In [ ]:
def get_order_status(order_id: str) -> str:
    """Look up the shipping status for a given order ID. Use this when the user asks about an order's status or tracking."""
    fake_db = {"98765": "Shipped, arriving in 2 days", "12345": "Delivered"}
    return fake_db.get(order_id, "Order not found")

print(get_order_status("98765"))


### TODO 3
Write one more tool function `initiate_refund(order_id: str) -> str` with a clear docstring, then write (in a markdown or comment) which of your two tools an agent should pick for the user message: *'Can I get my money back for order 98765?'* — and why.

In [ ]:
# TODO: define initiate_refund and write your reasoning as a comment


<details><summary>Discussion</summary>

The agent should pick `initiate_refund`, not `get_order_status`, because the user's intent ('get my money back') maps to a refund action, not a status lookup — this is exactly why clear, intent-specific docstrings matter for tool selection.
</details>

## Key Takeaways
- PromptTemplate + chains (`|`) let you compose LLM steps like a pipeline.
- Memory needs an explicit strategy (buffer, summary, limited) — it's not free or unbounded.
- Tool descriptions are the agent's only signal for picking the right tool — write them precisely.
- Understanding the pattern without LangChain (plain prompt-building) makes the abstraction click faster.
